In [0]:
df_bronze_lido = spark.read.table("bronze_cotacoes")

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_silver = (
    df_bronze_lido
    .drop("timestamp")
    .withColumn("bid", col("bid").cast("double"))
    .withColumn("high", col("high").cast("double"))
    .withColumn("low", col("low").cast("double"))
    .withColumn("pctChange", col("pctChange").cast("double"))
    .withColumn("create_date", to_timestamp(col("create_date")))
)

In [0]:
#df_silver.printSchema()

root
 |-- ask: string (nullable = true)
 |-- bid: double (nullable = true)
 |-- code: string (nullable = true)
 |-- codein: string (nullable = true)
 |-- create_date: timestamp (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- name: string (nullable = true)
 |-- pctChange: double (nullable = true)
 |-- varBid: string (nullable = true)



In [0]:
#display(df_silver)

ask,bid,code,codein,create_date,high,low,name,pctChange,varBid
5.0547,5.0525,USD,BRL,2026-10-19T15:21:42.000Z,5.065,5.0403,Dólar Americano/Real Brasileiro,-0.236941,-0.012
5.6683,5.6663,EUR,BRL,2026-10-19T15:19:53.000Z,5.698,5.6586,Euro/Real Brasileiro,-0.519671,-0.0296
5.0622,5.06,USD,BRL,2026-10-19T14:22:15.000Z,5.065,5.0403,Dólar Americano/Real Brasileiro,-0.088851,-0.0045
5.6724,5.6704,EUR,BRL,2026-10-19T14:19:52.000Z,5.698,5.6586,Euro/Real Brasileiro,-0.44769,-0.0255


In [0]:
df_silver_dedup = df_silver.dropDuplicates(["code", "create_date"])
#display(df_silver_dedup)

ask,bid,code,codein,create_date,high,low,name,pctChange,varBid
5.0547,5.0525,USD,BRL,2026-10-19T15:21:42.000Z,5.065,5.0403,Dólar Americano/Real Brasileiro,-0.236941,-0.012
5.6683,5.6663,EUR,BRL,2026-10-19T15:19:53.000Z,5.698,5.6586,Euro/Real Brasileiro,-0.519671,-0.0296
5.0622,5.06,USD,BRL,2026-10-19T14:22:15.000Z,5.065,5.0403,Dólar Americano/Real Brasileiro,-0.088851,-0.0045
5.6724,5.6704,EUR,BRL,2026-10-19T14:19:52.000Z,5.698,5.6586,Euro/Real Brasileiro,-0.44769,-0.0255


In [0]:
from pyspark.sql import functions as f

df_silver = df_silver.withColumn("status_risco", f.when(f.abs(col("pctChange")) >= 1.0, "ALTO RISCO").otherwise("NORMAL"))

#display(df_silver.select("code", "bid", "pctChange", "status_risco"))

code,bid,pctChange,status_risco
USD,5.0525,-0.236941,NORMAL
EUR,5.6663,-0.519671,NORMAL
USD,5.06,-0.088851,NORMAL
EUR,5.6704,-0.44769,NORMAL


In [0]:
df_silver.write.format("Delta").mode("overwrite").saveAsTable("silver_cotacoes")

In [0]:
%sql
-- SELECT * FROM silver_cotacoes

ask,bid,code,codein,create_date,high,low,name,pctChange,varBid,status_risco
5.0622,5.06,USD,BRL,2026-10-19T14:22:15.000Z,5.065,5.0403,Dólar Americano/Real Brasileiro,-0.088851,-0.0045,NORMAL
5.6724,5.6704,EUR,BRL,2026-10-19T14:19:52.000Z,5.698,5.6586,Euro/Real Brasileiro,-0.44769,-0.0255,NORMAL
5.0547,5.0525,USD,BRL,2026-10-19T15:21:42.000Z,5.065,5.0403,Dólar Americano/Real Brasileiro,-0.236941,-0.012,NORMAL
5.6683,5.6663,EUR,BRL,2026-10-19T15:19:53.000Z,5.698,5.6586,Euro/Real Brasileiro,-0.519671,-0.0296,NORMAL
